# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/Mvdu12/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

for col in ['word_count', 'days_since_last_update', 'impressions_90d', 'sessions_90d']:
    d = df[col].describe(percentiles=[.5, .75, .9, .99])
    print(f"--- {col} ---")
    print(d.round(1))
    print()

print(
    "Heavy tails: impressions_90d and sessions_90d are the extreme cases -- median "
    "731 impressions / 7 sessions vs a mean of 5,200 / 37, and a max of 517,715 / "
    "4,345. A handful of very visible pages pull the mean far above the typical "
    "page, so I'll lean on medians and bucket comparisons below rather than raw "
    "means. word_count is close to symmetric (median 2,877 vs mean 3,108). "
    "days_since_last_update is bimodal, not smoothly tailed -- it clusters near 20 "
    "days, then jumps to a 104-day plateau (75th and 90th percentile both land on "
    "104), which is exactly why the freshness_tier buckets below have such uneven "
    "sample sizes."
)


--- word_count ---
count    22301.0
mean      3107.8
std       1452.4
min          8.0
50%       2877.0
75%       3666.0
90%       5327.0
99%       7292.0
max       9546.0
Name: word_count, dtype: float64

--- days_since_last_update ---
count    30000.0
mean        46.1
std         42.1
min          1.0
50%         20.0
75%        104.0
90%        104.0
99%        106.0
max        373.0
Name: days_since_last_update, dtype: float64

--- impressions_90d ---
count     30000.0
mean       5200.4
std       16838.0
min           1.0
50%         731.0
75%        3615.2
90%       12136.4
99%       73505.8
max      517715.0
Name: impressions_90d, dtype: float64

--- sessions_90d ---
count    30000.0
mean        37.1
std        107.1
min          1.0
50%          7.0
75%         27.0
90%         88.0
99%        451.0
max       4345.0
Name: sessions_90d, dtype: float64

Heavy tails: impressions_90d and sessions_90d are the extreme cases -- median 731 impressions / 7 sessions vs a mean of 5,200 / 3

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# --- Signal test 1: staleness vs decline rate ---
signal1 = df.groupby('freshness_tier').agg(
    n=('content_id', 'size'), decline_rate=('is_declining', 'mean')
).round(3)
print("Signal 1 -- freshness_tier vs decline rate")
print(signal1)
print(
    "Verdict: CONFIRMED (on the two well-populated buckets) -- 91-180 days "
    "(n=9,171) declines at 61.1% vs 51.1% for 0-30 days (n=20,480). The 31-90 "
    "and 181+ buckets have only ~175 rows each -- too small to trust alone.\n"
)

# --- Signal test 2: word count vs traffic ---
signal2 = df.groupby('word_count_tier').agg(
    n=('content_id', 'size'), avg_sessions=('sessions_90d', 'mean'),
    median_sessions=('sessions_90d', 'median'),
).round(1)
print("Signal 2 -- word_count_tier vs sessions_90d")
print(signal2)
print(
    "Verdict: OPPOSITE -- 3500+ words averages 82.4 sessions (median 22.0, "
    "n=6,285) vs 2.1 (median 1.0, n=973) for under 1,000 words. Longer content "
    "gets MORE traffic here, not less.\n"
)

# --- Signal test 3: position vs CTR ---
signal3 = df.groupby('position_tier').agg(
    n=('content_id', 'size'), avg_ctr=('ctr', 'mean'), median_ctr=('ctr', 'median'),
).round(3)
# order by rank quality, not alphabetically
order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
signal3 = signal3.reindex(order)
print("Signal 3 -- position_tier vs CTR (ordered top_3 -> deep)")
print(signal3)
print(
    "Verdict: CONFIRMED -- average CTR falls in lockstep with rank quality: "
    "1.484 (top_3) -> 0.652 (page_1) -> 0.323 (striking) -> 0.222 (page_3_5) -> "
    "0.150 (deep). Note top_3's median CTR is 0.00 despite the high mean -- "
    "another heavy tail, a few very-high-CTR pages are pulling that average up."
)


Signal 1 -- freshness_tier vs decline rate
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
181+              174         0.471
31-90             175         0.589
91-180           9171         0.611
Verdict: CONFIRMED (on the two well-populated buckets) -- 91-180 days (n=9,171) declines at 61.1% vs 51.1% for 0-30 days (n=20,480). The 31-90 and 181+ buckets have only ~175 rows each -- too small to trust alone.

Signal 2 -- word_count_tier vs sessions_90d
                     n  avg_sessions  median_sessions
word_count_tier                                      
1000-2000         3780          12.4              4.0
2000-3500        11263          28.7              6.0
3500+             6285          82.4             22.0
<1000              973           2.1              1.0
Verdict: OPPOSITE -- 3500+ words averages 82.4 sessions (median 22.0, n=6,285) vs 2.1 (median 1.0, n=973) for under 1,000 words. Longer content gets MORE traf

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# Signal 1 (staleness) is the one tied to a real FlyRank flag: the refresh flag
# assumes "not updated in a while" predicts decline risk. Testing that assumption
# a level deeper than the tier table above, on raw days_since_last_update instead
# of the (unevenly populated) freshness_tier buckets.

median_days = df['days_since_last_update'].median()
above = df[df['days_since_last_update'] > median_days]
below = df[df['days_since_last_update'] <= median_days]

print(f"Median days_since_last_update: {median_days:.0f}")
print(
    f"Above median ({len(above):,} rows): decline rate = {above['is_declining'].mean():.3f}"
)
print(
    f"At/below median ({len(below):,} rows): decline rate = {below['is_declining'].mean():.3f}"
)

# also check within just the two well-populated freshness_tier buckets, since
# those carry real sample size (Section 2's 31-90 / 181+ buckets do not)
big_buckets = df[df['freshness_tier'].isin(['0-30', '91-180'])]
t = big_buckets.groupby('freshness_tier')['is_declining'].agg(['size', 'mean']).round(3)
print("\nSame check, restricted to the two buckets with real sample size:")
print(t)

print(
    "\nVerdict: CONFIRMED, and it holds at both the median-split level and the "
    "bucket level -- FlyRank's refresh flag is leaning on a real (if moderate) "
    "signal, not a false one. The effect is directional (roughly +10 points of "
    "decline rate), not deterministic -- most fresh pages still decline sometimes, "
    "and most stale pages still don't."
)


Median days_since_last_update: 20
Above median (14,134 rows): decline rate = 0.546
At/below median (15,866 rows): decline rate = 0.539

Same check, restricted to the two buckets with real sample size:
                 size   mean
freshness_tier              
0-30            20480  0.511
91-180           9171  0.611

Verdict: CONFIRMED, and it holds at both the median-split level and the bucket level -- FlyRank's refresh flag is leaning on a real (if moderate) signal, not a false one. The effect is directional (roughly +10 points of decline rate), not deterministic -- most fresh pages still decline sometimes, and most stale pages still don't.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can trust staleness and position as real, directional signals for prioritization -- stale pages decline somewhat more often, and CTR tracks position cleanly, so both are safe inputs for a refresh-priority rule. Word count should NOT be used as a standalone signal for *this page is too long* -- the data says the opposite of the intuitive story. And because impressions/sessions are heavily right-skewed, any team dashboard reporting *average traffic* per segment should show the median alongside it, or a handful of outlier pages will make every segment look busier than it is.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.